In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *
from utils import process_greek

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Greece Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'GR'
NUTS2 = 'Thessaly'

In [4]:
YEAR = 2023
MONTH = 'September'
PERIOD = '2nd'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model_new.pkl', 'rb'))
scaler = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Scaler_new.pkl', 'rb'))
imputer = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Imputer_new.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,area,population,population_density,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,22.76105,39.69186,2023-09-16,θεσσαλιας,αγιας,16,9,37,2023,-0.101168,-0.994869,-1.0,-1.836970e-16,-0.947326,-0.32027,661.8,10705.0,17.3,45.75488,0.581514,0.179964,-0.495005,-0.179964,0.581337,0.188170,-0.494508,-0.188170,0.050713,0.057891,0.033468,0.057891,20.905000,25.106875,16.703125,10.178337,3.943677,10.285211,3.423839,16.579516,6.033934,17.971971,7.699233,0.0000,711.337289,1208.654188,8282.127022,1331.619869,1,123.585589,121.767859,183.036108,0.0,21.311709,31,90,30,90.0,30,90,10,10,1,6,6,2,0,1,0,0
1,22.72671,39.12560,2023-09-16,θεσσαλιας,αλμυρου,16,9,37,2023,-0.101168,-0.994869,-1.0,-1.836970e-16,-0.947326,-0.32027,905.4,16004.0,20.6,45.24728,0.426989,0.135515,-0.369994,-0.135515,0.413289,0.128376,-0.354341,-0.128376,0.078060,0.067076,0.060420,0.067076,20.748750,25.812500,15.685000,11.191823,3.758937,10.731281,3.006374,16.563729,6.106926,19.041462,7.924166,0.0000,715.466976,1473.802352,10017.207897,1610.666116,14,259.585198,342.571837,171.163634,0.0,3.415750,11,80,10,72.0,10,72,1,1,7,1,1,2,0,1,0,0
2,23.99842,39.24585,2023-09-16,θεσσαλιας,αλοννησου,16,9,37,2023,-0.101168,-0.994869,-1.0,-1.836970e-16,-0.947326,-0.32027,129.6,3153.0,21.2,46.00175,-0.204052,0.309524,0.394273,-0.309524,-0.201121,0.306133,0.390697,-0.306133,0.003947,0.006961,0.004915,0.006961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,627.175109,1629.666833,1484.683259,55302.082778,24,302.637680,165.397106,195.027810,0.0,1.676994,21,94,20,94.0,20,94,8,8,4,1,1,2,0,1,0,0
3,21.48510,39.29630,2023-09-16,θεσσαλιας,αργιθεας,16,9,37,2023,-0.101168,-0.994869,-1.0,-1.836970e-16,-0.947326,-0.32027,372.9,3515.0,9.3,44.78626,0.545351,0.216142,-0.476508,-0.216142,0.536167,0.201358,-0.463853,-0.201358,0.085362,0.078157,0.055163,0.078157,18.168462,21.997692,14.339231,6.988053,1.639104,7.062325,-0.605892,13.120876,3.493461,14.383348,4.274521,0.0000,501.213224,1270.714386,19327.278832,1925.318354,27,284.268894,1127.254453,234.678996,0.0,1.719393,21,88,20,88.0,20,88,8,8,4,1,1,2,0,1,0,0
4,22.93502,39.38117,2023-09-16,θεσσαλιας,βολου,16,9,37,2023,-0.101168,-0.994869,-1.0,-1.836970e-16,-0.947326,-0.32027,385.6,138865.0,374.6,45.57293,0.403414,0.053834,-0.367204,-0.053834,0.386054,0.034504,-0.358844,-0.034504,0.066786,0.060885,0.049579,0.060885,21.549167,26.756667,16.341667,12.043947,4.126457,11.394699,3.845437,17.241617,5.836050,19.956371,8.057589,0.0295,1068.301826,1677.648732,5935.142744,3071.988689,5,143.181190,234.931813,174.426021,0.0,4.643743,31,91,30,91.0,30,91,10,10,1,6,6,2,0,1,0,0


In [7]:
features_to_remove = ['x', 'y', 'eq_distance','day', 'month', 'week', 'year', 'lc_prop1_assessment',
                    'lc_prop2', 'lc_prop2_assessment', 'lc_prop3', 'lc_prop3_assessment',
                    'lc_type2', 'lc_type3', 'lc_type4', 'lc_type5', 'lw', 'qc','ndvi_mean', 'ndmi_mean', 'ndwi_mean', 'ndbi_mean', 'ndvi_std',
                    'ndmi_std', 'ndwi_std', 'ndbi_std',]

In [8]:
object_cols = data_test.select_dtypes(include=['object']).columns.to_list()
removed_cols = features_to_remove

X_test = data_test.drop(columns = ['case'] + object_cols + removed_cols)
y_test = data_test['case']

X_test = scaler.transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)

In [9]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,22.15952,39.95174,ελασσονας,16,9,2023,0.349103
1,22.34107,39.61048,λαρισαιων,16,9,2023,0.165717
2,22.54288,39.85090,τεμπων,16,9,2023,0.128539
3,21.48510,39.29630,αργιθεας,16,9,2023,0.126577
4,22.29611,39.76297,τυρναβου,16,9,2023,0.075326
5,22.51098,39.51390,κιλελερ,16,9,2023,0.045409
6,21.89535,39.26951,καρδιτσας,16,9,2023,0.030032
7,21.49996,39.73065,καλαμπακας,16,9,2023,0.027000
8,21.77435,39.61293,τρικκαιων,16,9,2023,0.026915
9,22.43796,39.30328,φαρσαλων,16,9,2023,0.025468


In [10]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.0080309448997414, 0.0860446074909823, 0.5109550767195234, 0.8201371322355278, 0.9305335064919854, 1.0]


In [11]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,22.15952,39.95174,ελασσονας,16,9,2023,0.349103,2
1,22.34107,39.61048,λαρισαιων,16,9,2023,0.165717,2
2,22.54288,39.85090,τεμπων,16,9,2023,0.128539,2
3,21.48510,39.29630,αργιθεας,16,9,2023,0.126577,2
4,22.29611,39.76297,τυρναβου,16,9,2023,0.075326,1
5,22.51098,39.51390,κιλελερ,16,9,2023,0.045409,1
6,21.89535,39.26951,καρδιτσας,16,9,2023,0.030032,1
7,21.49996,39.73065,καλαμπακας,16,9,2023,0.027000,1
8,21.77435,39.61293,τρικκαιων,16,9,2023,0.026915,1
9,22.43796,39.30328,φαρσαλων,16,9,2023,0.025468,1


In [12]:
# results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}.csv", encoding = enc, index = False)

In [13]:
##TODO Visualisation of results